**Nama: Yuma Hazza Yuditama** <br>
**NIM: 24060124120035** <br>
**Lab: C1** <br>
**Materi: Klasifikasi** <br>

# Latihan

In [54]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42

1. Load ulang dataset Churn Modelling dengan nama variabel df_latihan!
   -

In [55]:
DATA_PATH = "/kaggle/input/datasets/shrutimechlearn/churn-modelling/Churn_Modelling.csv"

df_latihan = pd.read_csv(DATA_PATH)

print("Ukuran dataset:", df_latihan.shape)
display(df_latihan.head())

print("\n")
print("Cek informasi dataset:")
df_latihan.info()

Ukuran dataset: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0




Cek informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


2. Lakukan feature selection dan data splitting dengan proporsi 70:30.
   -

In [56]:
# Feature Selection
fitur_numerik = df_latihan.drop(columns=['RowNumber', 'CustomerId']).select_dtypes(include=['int64', 'float64'])
matriks_corr = fitur_numerik.corr()

kolom_dihapus = ["RowNumber", "CustomerId", "Surname"]
kolom_dihapus = [kolom for kolom in kolom_dihapus if kolom in df_latihan.columns]
kolom_target = "Exited"

# Data Splitting
X = df_latihan.drop(columns=kolom_dihapus + [kolom_target])
y = df_latihan[kolom_target].astype(int)

print("Fitur yang digunakan:")
print(X.columns.tolist())
print("\nUkuran X:", X.shape)
print("Ukuran y:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30,random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nProporsi kelas pada data latih:")
display(y_train.value_counts().sort_index())

print("\nProporsi kelas pada data uji:")
display(y_test.value_counts().sort_index())

Fitur yang digunakan:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

Ukuran X: (10000, 10)
Ukuran y: (10000,)
X_train: (7000, 10)
X_test: (3000, 10)

Proporsi kelas pada data latih:


Exited
0    5574
1    1426
Name: count, dtype: int64


Proporsi kelas pada data uji:


Exited
0    2389
1     611
Name: count, dtype: int64

3. Buatlah pipeline preprocessing! Berikan justifikasi pemilihan metode preprocessing!
   -

In [57]:
fitur_numerik = X_train.select_dtypes(include=['number']).columns.tolist()
fitur_kategori = X_train.select_dtypes(exclude=['number']).columns.tolist()

age_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])


numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("age", age_pipeline, ["Age"]),
        ("num", numeric_pipeline, [
            col for col in numeric_features if col != "Age"
        ]),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)
display(preprocessor)



ValueError: at least one of include or exclude must be nonempty

4. Buatlah kode GridSearchCV dengan 3-fold Cross Validation untuk model machine learning selain Logistic Regression dan Random Forest! Pilihlah minimal 3 model dan minimal 3 jenis parameter! Berikan justifikasi untuk: pemilihan model, pemilihan jenis parameter, dan penentuan ruang pemilihan parameter!
   -

In [ ]:
# Jawab : . . .




5. Jalankan hyperparameter tuning yang sudah disiapkan oleh jawaban soal nomor 4.
   -

In [ ]:
# Jawab : . . .




6. Prediksi data uji dengan menggunakan parameter terbaik tiap model!
   -

In [ ]:
# Jawab : . . .




7. Analisislah model mana yang paling baik dan tentukan metrik evaluasi utama yang digunakan! Berikan justifikasinya!
   -

Jawab:
<br>
. . . . .